# Kaggle remote run — the tuned harness on the competition checkpoint

This notebook is the first **remote** execution of the harness we have been tuning
locally. It follows the official **Getting Started: Gemma 4 Developer Agent**
notebook: it installs the competition wheelhouse, loads the competition data from
`/kaggle/input/competitions/gemma-4-developer-agent`, serves the checkpoint with
vLLM through `adk_submission`, and runs `swegemma`'s `Evaluator` with the
Docker-less `subprocess` sandbox. It then archives the artifacts with the same
`runs/<UTC>/report.json` + `STATUS` handoff the host trial produces.

## Before you run

1. **Accelerator** — Settings → Accelerator → `GPU L4 x4` (the scored environment).
   `GPU T4 x2` is not representative and the W4A16 checkpoint may not load on Turing.
2. **Internet** — Settings → Internet → `On`. The notebook clones this repository
   and installs the wheelhouse.
3. **Inputs** (Add Input):
   - Competition data `gemma-4-developer-agent` (mounted at
     `/kaggle/input/competitions/gemma-4-developer-agent`).
   - Dataset `metric/gemma-4-developer-agent-wheelhouse`.
   - Model `google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct` (or the family and
     version you are running; the notebook finds it under
     `/kaggle/input/models/google/gemma-4/`).
4. **Secrets** — only for `MODEL_BACKEND = "api"`: add `OPENAI_API_KEY`
   (Add-ons → Secrets). The `local` backend needs no secret.

Run the cells top to bottom. The configuration cell is the only one you edit for a
normal run; the last cells archive, package and stop the model server.


## 1. Configuration

In [ ]:
# =============================================================================
# 1. Configuration — edit this cell, then run the notebook top to bottom.
# =============================================================================
RUN_LABEL = "kaggle_remote_01"

# Repository that carries the tuned harness (submission/, experiments/subsets/).
REPO_URL = "https://github.com/emiliodavola/kaggle-gemma-agent.git"
REPO_REF = "main"  # a branch, tag or commit sha; pin one for a reproducible run

# What to evaluate.
SUBSET = "dev-v1"  # experiments/subsets/<id>.json; None disables it
TASK_IDS: list[str] = []  # explicit ids; wins over SUBSET
MAX_TASKS = 2  # the first remote run stays small on purpose

# Budgets. None means "use submission/eval_config.yaml" (the tuned tree).
# MAX_TIME_MINUTES = 30 is a first-run cap; the scored budget is 60.
MAX_TOOL_CALLS: int | None = None
MAX_TIME_MINUTES: float | None = 30
MAX_TURNS: int | None = None
COMMAND_TIMEOUT_SECONDS: int | None = None
CONCURRENCY = 1
DISPLAY = "quiet"  # quiet | single | dashboard | auto

# Model backend:
#   "local" — vLLM on this notebook's GPUs, serving the model declared in agent.yaml.
#   "api"   — an OpenAI-compatible endpoint (Kaggle secret OPENAI_API_KEY).
MODEL_BACKEND = "local"
MODEL_PATH = ""  # "" = find it under /kaggle/input/models/google/gemma-4
TENSOR_PARALLEL_SIZE = 0  # 0 = 4/2/1 by visible GPU count
MAX_MODEL_LEN = 32768  # the scored context ceiling
GPU_MEMORY_UTILIZATION = 0.80  # the scoring value; the getting-started notebook uses 0.90
VLLM_DTYPE = "auto"  # auto picks bfloat16 when the GPU supports it
VLLM_EXTRA_ARGS: list[str] = []

# "api" backend only.
API_PRESET = "opencode"  # see BACKEND_PRESETS in scripts/host-trial/run_host_trial.py
API_MODEL = "deepseek-v4.1-flash"
API_BASE_URL = ""  # "" = the preset URL

# Paths. Keep the defaults unless your inputs are mounted elsewhere.
DATA_ROOT = ""  # "" = auto-detect under /kaggle/input
WHEELHOUSE = ""  # "" = auto-detect the swegemma wheelhouse
WORK_DIR = "/kaggle/working/kaggle-remote-run"

if MODEL_BACKEND not in {"local", "api"}:
    raise ValueError("MODEL_BACKEND must be 'local' or 'api'")
if MAX_TASKS < 1:
    raise ValueError("MAX_TASKS must be at least 1")
print("run label :", RUN_LABEL)
print("backend   :", MODEL_BACKEND)
print("selection :", TASK_IDS or SUBSET or "whole corpus", "| cap", MAX_TASKS)

## 2. Environment probe

In [ ]:
# =============================================================================
# 2. Environment probe — same switches as the official getting-started notebook.
# =============================================================================
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)

for name, value in {
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_FLAX": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
    "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    "NCCL_DEBUG": "WARN",
    "LITELLM_LOCAL_MODEL_COST_MAP": "True",
    "LITELLM_TELEMETRY": "False",
    "LITELLM_LOG": "WARNING",
    "LOCAL_API_KEY": "EMPTY",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "OTEL_SDK_DISABLED": "true",
    "OTEL_PYTHON_DISABLED": "true",
}.items():
    os.environ.setdefault(name, value)


def sh(argv, **kwargs):
    # Run a command, echo it, and return the CompletedProcess.
    print("$ " + " ".join(str(a) for a in argv), flush=True)
    kwargs.setdefault("text", True)
    return subprocess.run([str(a) for a in argv], **kwargs)


def bounded_glob(root, pattern, max_depth=3):
    # Yield files matching pattern up to max_depth directory levels below root.
    # Symlinks are followed with cycle protection: Kaggle input mounts can be links.
    root = Path(root)
    if not root.is_dir():
        return
    seen = set()
    stack = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            resolved = current.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        try:
            entries = list(current.iterdir())
        except (PermissionError, OSError):
            continue
        for entry in entries:
            if entry.is_file() and entry.match(pattern):
                yield entry
            elif entry.is_dir() and depth < max_depth:
                stack.append((entry, depth + 1))


def accepted_fields(cls):
    # Dataclass or pydantic model: the field names this harness version accepts.
    try:
        import dataclasses

        return {field.name for field in dataclasses.fields(cls)}
    except TypeError:
        return set(getattr(cls, "model_fields", {}))


print("python :", sys.version.split()[0], "|", sys.executable)
print("cwd    :", os.getcwd())

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print("gpus   :\n" + (smi.stdout.strip() or smi.stderr.strip() or "none"))

usage = shutil.disk_usage(WORK)
print(f"work   : {usage.free / 2**30:.1f} GiB free of {usage.total / 2**30:.1f} GiB at {WORK}")

## 3. Install the wheelhouse

In [ ]:
# =============================================================================
# 3. Wheelhouse install — the official path: no dependency resolution.
# =============================================================================
import glob
import importlib


def unpack_wheelhouse(archive):
    target = WORK / "wheelhouse"
    if list(target.glob("swegemma-*.whl")):
        return target
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(target)
    print(f"unpacked {archive} -> {target}")
    return target


def walk_files(root, max_depth):
    # Yield every file under root, following symlinks with cycle protection.
    root = Path(root)
    seen = set()
    stack = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            resolved = current.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        try:
            entries = list(current.iterdir())
        except (PermissionError, OSError):
            continue
        for entry in entries:
            if entry.is_file():
                yield entry
            elif entry.is_dir() and depth < max_depth:
                stack.append((entry, depth + 1))


def input_tree(root="/kaggle/input", max_depth=3):
    # Render the mounted input tree (directories plus a file sample) for diagnostics.
    lines = []
    seen = set()

    def visit(directory, depth):
        if depth > max_depth:
            return
        try:
            resolved = directory.resolve()
        except OSError:
            return
        if resolved in seen:
            return
        seen.add(resolved)
        try:
            entries = sorted(directory.iterdir(), key=lambda path: path.name.lower())
        except (PermissionError, OSError):
            return
        files = [entry for entry in entries if entry.is_file()]
        directories = [entry for entry in entries if entry.is_dir()]
        if files:
            sample = ", ".join(entry.name for entry in files[:4])
            more = f", +{len(files) - 4} more" if len(files) > 4 else ""
            lines.append("  " * depth + f"[{len(files)} files: {sample}{more}]")
        for entry in directories:
            lines.append("  " * depth + entry.name + "/")
            visit(entry, depth + 1)

    visit(Path(root), 0)
    return "\n".join(lines)


def find_wheelhouse():
    if WHEELHOUSE:
        candidate = Path(WHEELHOUSE)
        if candidate.is_dir():
            for wheel in candidate.rglob("swegemma-*.whl"):
                return wheel.parent
        if candidate.is_file() and candidate.suffix == ".zip":
            return unpack_wheelhouse(candidate)
        raise FileNotFoundError(f"WHEELHOUSE does not hold swegemma-*.whl: {candidate}")
    input_root = Path("/kaggle/input")
    if not input_root.is_dir() or not any(input_root.iterdir()):
        raise FileNotFoundError(
            "No inputs are attached to this notebook session: /kaggle/input is "
            "empty. Attach the competition data, the wheelhouse dataset and the "
            "model in Add Input, then re-run this cell."
        )
    for entry in walk_files("/kaggle/input", 6):
        name = entry.name.lower()
        if name.startswith("swegemma") and name.endswith(".whl"):
            return entry.parent
    for entry in walk_files("/kaggle/input", 6):
        name = entry.name.lower()
        if "wheelhouse" in name and name.endswith(".zip"):
            return unpack_wheelhouse(entry)
    raise FileNotFoundError(
        "No swegemma wheelhouse under /kaggle/input. Attach the dataset "
        "'metric/gemma-4-developer-agent-wheelhouse' or set WHEELHOUSE.\n"
        "/kaggle/input contains:\n" + input_tree()
    )


WHEELHOUSE_DIR = find_wheelhouse()
print("wheelhouse:", WHEELHOUSE_DIR)
if sys.version_info[:2] != (3, 12):
    print(
        "warning: the competition wheelhouse is built for CPython 3.12; "
        f"this session runs {sys.version.split()[0]}"
    )

# Broken cutlass .pth hooks shipped by the base image.
for pth_pattern in (
    "/usr/local/lib/python*/dist-packages/*cutlass*.pth",
    "/usr/local/lib/python*/site-packages/*cutlass*.pth",
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Kaggle dataset uploads strip the PEP 440 '+cu128' local version; restore it.
staged = Path("/tmp/wheelhouse")
staged.mkdir(parents=True, exist_ok=True)
for wheel in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in wheel.name.lower():
        continue
    target_name = (
        wheel.name.replace("cu128", "+cu128")
        if ("cu128" in wheel.name and "+" not in wheel.name)
        else wheel.name
    )
    target = staged / target_name
    if not target.exists():
        try:
            os.symlink(wheel, target)
        except OSError:
            shutil.copyfile(wheel, target)

wheels = sorted(str(path) for path in staged.glob("*.whl"))
print(f"installing {len(wheels)} wheels from {WHEELHOUSE_DIR}")
sh(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "--force-reinstall",
        *wheels,
    ],
    check=True,
)
importlib.invalidate_caches()

# The wheelhouse is a delta over the image. Fill a genuinely missing runtime
# dependency from PyPI; the pinned wheels above still win when they exist.
RUNTIME_DEPS = {
    "litellm": "litellm>=1.83.7,<=1.83.14",
    "docker": "docker>=7.0.0",
    "pandas": "pandas>=2.0.0",
    "networkx": "networkx>=3.0",
    "yaml": "pyyaml>=6.0.0",
    "dotenv": "python-dotenv>=1.0.0",
    "cachetools": "cachetools>=5.0.0",
    "rich": "rich>=13.0.0",
    "google.genai": "google-genai>=2.9,<3",
    "requests": "requests",
}
missing = []
for module, spec in RUNTIME_DEPS.items():
    try:
        importlib.import_module(module)
    except Exception:
        missing.append(spec)
if missing:
    print("installing missing runtime deps:", missing)
    sh([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()

if MODEL_BACKEND == "local":
    for module in ("torch", "vllm"):
        try:
            importlib.import_module(module)
        except Exception as exc:
            raise RuntimeError(f"{module} is required for the local backend: {exc}") from exc

for dist in (
    "swegemma",
    "adk-submission",
    "adk-eval-core",
    "google-adk",
    "litellm",
    "transformers",
    "torch",
    "vllm",
):
    try:
        print(f"{dist:<15} {importlib.metadata.version(dist)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{dist:<15} MISSING")

## 4. Clone the repository and run the submission gate

In [ ]:
# =============================================================================
# 4. Repository + submission gate — the same gate CI runs, before any GPU time.
# =============================================================================
REPO_DIR = WORK / "repo"
if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    sh(["git", "-C", REPO_DIR, "init", "--quiet"], check=True)
    sh(["git", "-C", REPO_DIR, "remote", "add", "origin", REPO_URL], check=True)

sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_REF], check=True)
sh(["git", "-C", REPO_DIR, "checkout", "--force", "FETCH_HEAD"], check=True)

COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print("repo commit:", COMMIT)

GATE_ENV = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src")}
gate = sh(
    [sys.executable, "-m", "kaggle_gemma_agent.pack", "submission", "--check"],
    cwd=REPO_DIR,
    env=GATE_ENV,
)
if gate.returncode != 0:
    raise SystemExit("submission gate failed: fix submission/ before spending GPU time")

SUBMISSION_DIR = REPO_DIR / "submission"
print("submission:", SUBMISSION_DIR)

## 5. Competition data and task selection

In [ ]:
# =============================================================================
# 5. Competition data + task selection — validates only the tasks this run needs.
# =============================================================================
def find_data_root():
    if DATA_ROOT:
        return Path(DATA_ROOT)
    for candidate in (
        "/kaggle/input/competitions/gemma-4-developer-agent",
        "/kaggle/input/gemma-4-developer-agent",
        "/kaggle/input/competition-data",
    ):
        if (Path(candidate) / "tasks.jsonl").is_file():
            return Path(candidate)
    candidates = []
    for tasks in bounded_glob("/kaggle/input", "tasks.jsonl", max_depth=3):
        root = tasks.parent
        if (root / "snapshots").is_dir() or (root / "wheels").is_dir():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "No competition data root under /kaggle/input (need tasks.jsonl next to "
            "snapshots/ and wheels/). Attach the competition data or set DATA_ROOT."
        )
    return sorted(candidates, key=lambda path: len(str(path)))[0]


DATA = find_data_root()
TASKS_FILE = DATA / "tasks.jsonl"
SNAPSHOTS_DIR = DATA / "snapshots"
WHEELS_COMPETITION = DATA / "wheels"
GRAPHS_DIR = DATA / "graphs"
EMBEDDINGS_DIR = DATA / "embeddings"
print("data root:", DATA)

missing = [path for path in (TASKS_FILE, SNAPSHOTS_DIR, WHEELS_COMPETITION) if not path.exists()]
if missing:
    raise FileNotFoundError(
        "competition data is incomplete; missing: " + ", ".join(str(p) for p in missing)
    )
if not GRAPHS_DIR.is_dir() or not EMBEDDINGS_DIR.is_dir():
    print("warning: graphs/ or embeddings/ missing; the code-intelligence tools degrade")

corpus = {}
with TASKS_FILE.open(encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            record = json.loads(line)
            corpus[record["instance_id"]] = record

if TASK_IDS:
    selected = list(TASK_IDS)
elif SUBSET:
    subset_path = REPO_DIR / "experiments" / "subsets" / f"{SUBSET}.json"
    selected = json.loads(subset_path.read_text(encoding="utf-8"))["task_ids"]
else:
    selected = list(corpus)
selected = selected[:MAX_TASKS]
if not selected:
    raise ValueError("no task selected; set TASK_IDS, SUBSET or MAX_TASKS")

unknown = [task_id for task_id in selected if task_id not in corpus]
if unknown:
    raise ValueError(f"task ids not in tasks.jsonl: {unknown}")

absent = []
for task_id in selected:
    if not any((SNAPSHOTS_DIR / f"{task_id}{suffix}").is_file() for suffix in (".tgz", ".tar.gz")):
        absent.append(task_id)
if absent:
    raise FileNotFoundError(
        "snapshots missing for "
        + ", ".join(absent)
        + ". Attach a dataset with those snapshots or fetch them with "
        + "'kaggle competitions download -c gemma-4-developer-agent -f snapshots/<id>.tgz'."
    )

print(f"tasks ({len(selected)}):")
for task_id in selected:
    record = corpus[task_id]
    print(f"  {task_id:<18} {record['repo']:<18} {record['base_commit'][:10]}")

## 6. Start the model

In [ ]:
# =============================================================================
# 6. Model backend — vLLM with the scoring wiring, or an OpenAI-compatible API.
# =============================================================================
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models import setup_gemma_model_registry

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset(
    {
        "gemma-4-31b-it-qat-w4a16-ct",
        "gemma-4-12b-it-qat-w4a16-ct",
        "gemma-4-e4b-it-qat-w4a16-ct",
        "gemma-4-e2b-it-qat-w4a16-ct",
    }
)
MODEL_VLLM_EXTRA_ARGS = {
    "gemma-4-12b-it-qat-w4a16-ct": (
        "--hf-overrides",
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
SAFE_PATH_COMPONENT = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]*")

declared = discover_declared_models(SUBMISSION_DIR)
if len(declared) != 1:
    raise ValueError(f"expected exactly one declared model, got {sorted(declared)}")
DECLARED_MODEL = next(iter(declared))
if DECLARED_MODEL not in ALLOWED_MODEL_NAMES:
    raise ValueError(
        f"unsupported model {DECLARED_MODEL!r}; allowed: {sorted(ALLOWED_MODEL_NAMES)}"
    )
print("declared model:", DECLARED_MODEL)

ADAPTERS = discover_adapters(str(SUBMISSION_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(ADAPTERS, "adapters", None) or {}
for adapter_name in adapter_map:
    if not SAFE_PATH_COMPONENT.fullmatch(adapter_name):
        raise ValueError(f"unsafe adapter name {adapter_name!r}")
detected_ranks = [rank for info in adapter_map.values() if (rank := getattr(info, "rank", None))]
print("adapters:", sorted(adapter_map) or "none")

MODEL_SERVER = None
HEADER_PROXY = None
API_KEY = "EMPTY"


def secret(name):
    value = os.environ.get(name, "")
    if value and value != "EMPTY":
        return value
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""


def find_model_dir(model_name):
    if MODEL_PATH:
        return Path(MODEL_PATH)
    base = Path("/kaggle/input/models/google/gemma-4")
    for family in ("other", "Other", "transformers", "Transformers"):
        family_dir = base / family / model_name
        if not family_dir.is_dir():
            continue
        versions = [
            path
            for path in family_dir.iterdir()
            if path.is_dir() and (path / "config.json").is_file()
        ]
        if versions:

            def version_key(path):
                return (1, int(path.name)) if path.name.isdigit() else (0, path.name)

            return max(versions, key=version_key)
    found = [
        path.parent
        for path in bounded_glob("/kaggle/input", "config.json", max_depth=6)
        if model_name.lower() in str(path.parent).lower()
    ]
    if found:
        return found[0]
    raise FileNotFoundError(
        f"model {model_name} not found under /kaggle/input. Attach the Kaggle Model "
        f"'google/gemma-4/other/{model_name}' (or its transformers/ family) or set MODEL_PATH."
    )


if MODEL_BACKEND == "local":
    model_dir = find_model_dir(DECLARED_MODEL)
    if not torch.cuda.is_available():
        raise RuntimeError("MODEL_BACKEND='local' needs a GPU accelerator; none is visible")
    gpu_count = torch.cuda.device_count()
    parallel = TENSOR_PARALLEL_SIZE or (4 if gpu_count >= 4 else 2 if gpu_count >= 2 else 1)
    dtype = VLLM_DTYPE
    if dtype == "auto" and torch.cuda.is_bf16_supported():
        dtype = "bfloat16"
    print("model path:", model_dir)
    print(f"gpus: {gpu_count} | tensor_parallel_size: {parallel} | dtype: {dtype}")

    vllm_kwargs = dict(
        model=str(model_dir),
        port=8000,
        host="127.0.0.1",
        tool_call_parser="gemma4",
        reasoning_parser="gemma4",
        max_model_len=MAX_MODEL_LEN,
        dtype=dtype,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        enable_auto_tool_choice=True,
        enable_lora=bool(adapter_map),
        max_loras=max(1, len(adapter_map)),
        max_lora_rank=max(detected_ranks) if detected_ranks else 128,
        scheduler_reserve_full_isl=False,
        tensor_parallel_size=parallel,
        startup_timeout=1200,
        extra_args=MODEL_VLLM_EXTRA_ARGS.get(DECLARED_MODEL, ()) + tuple(VLLM_EXTRA_ARGS),
    )
    accepted = accepted_fields(VllmConfig)
    ignored = sorted(set(vllm_kwargs) - accepted)
    vllm_kwargs = {key: value for key, value in vllm_kwargs.items() if key in accepted}
    if ignored:
        print("ignored by this VllmConfig:", ignored)

    MODEL_SERVER = VllmServer(VllmConfig(**vllm_kwargs), adapter_manifest=ADAPTERS)
    MODEL_SERVER.start()
    BASE_URL = MODEL_SERVER.base_url
    MODELS = MODEL_SERVER.create_model_registry(
        aliases=[DECLARED_MODEL], model_prefix="openai/", api_key="EMPTY"
    )
    MODEL_ID = DECLARED_MODEL
    BACKEND_NAME = "vllm-kaggle"
else:
    sys.path.insert(0, str(REPO_DIR / "scripts" / "host-trial"))
    from run_host_trial import (
        BACKEND_PRESETS,
        DEFAULT_TRIAL_SESSION,
        HeaderInjectingProxy,
        smoke_test_backend,
    )

    API_KEY = secret("OPENAI_API_KEY")
    if not API_KEY:
        raise RuntimeError(
            "MODEL_BACKEND='api' needs an OPENAI_API_KEY Kaggle secret "
            "(Add-ons -> Secrets) or the OPENAI_API_KEY environment variable."
        )
    upstream = API_BASE_URL or BACKEND_PRESETS[API_PRESET]
    HEADER_PROXY = HeaderInjectingProxy(
        upstream, secret("HARNESS_TRIAL_SESSION") or DEFAULT_TRIAL_SESSION
    )
    BASE_URL = HEADER_PROXY.start()
    print("upstream:", upstream)
    print("smoke:", smoke_test_backend(BASE_URL, API_MODEL, API_KEY))
    MODELS = setup_gemma_model_registry(api_base=BASE_URL, api_key=API_KEY, served_model=API_MODEL)
    MODEL_ID = API_MODEL
    BACKEND_NAME = API_PRESET

print("base url:", BASE_URL)
print("model id:", MODEL_ID)

## 7. Evaluate

In [ ]:
# =============================================================================
# 7. Evaluation — Evaluator API with the Docker-less subprocess sandbox.
# =============================================================================
import asyncio
import concurrent.futures

import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coroutine):
    # Run a coroutine from a notebook cell, with or without a live event loop.
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coroutine)).result()
    return asyncio.run(coroutine)


eval_config_file = next(
    (
        path
        for path in (
            SUBMISSION_DIR / "eval_config.yaml",
            SUBMISSION_DIR / "eval_config.yml",
        )
        if path.is_file()
    ),
    None,
)
raw_eval_config = (
    yaml.safe_load(eval_config_file.read_text(encoding="utf-8")) or {}
    if eval_config_file is not None
    else {}
)
eval_section = raw_eval_config.get("evaluation", raw_eval_config)

timeout_seconds = int(eval_section.get("timeout_seconds", 300))
verification_timeout_seconds = int(eval_section.get("verification_timeout_seconds", 300))
max_tool_calls = int(eval_section.get("max_tool_calls", 100))
max_time_minutes = float(eval_section.get("max_time_minutes", 60.0))
turns_raw = eval_section.get("max_turns", eval_section.get("max_llm_calls"))
max_turns = int(turns_raw) if turns_raw is not None else None

if COMMAND_TIMEOUT_SECONDS is not None:
    timeout_seconds = int(COMMAND_TIMEOUT_SECONDS)
if MAX_TOOL_CALLS is not None:
    max_tool_calls = int(MAX_TOOL_CALLS)
if MAX_TIME_MINUTES is not None:
    max_time_minutes = float(MAX_TIME_MINUTES)
if MAX_TURNS is not None:
    max_turns = int(MAX_TURNS)
print(
    "budgets:"
    f" tool_calls={max_tool_calls}"
    f" time={max_time_minutes:g}min"
    f" turns={max_turns}"
    f" command_timeout={timeout_seconds}s"
)

RESULTS_DIR = WORK / "results" / RUN_LABEL
if RESULTS_DIR.exists():
    raise SystemExit(
        f"{RESULTS_DIR} already exists; change RUN_LABEL so two runs never share a results dir"
    )

limits, generation_constraints = build_submission_limits()
eval_kwargs = dict(
    tasks_path=TASKS_FILE,
    snapshots_dir=SNAPSHOTS_DIR,
    results_dir=RESULTS_DIR,
    submission_dir=SUBMISSION_DIR,
    models=MODELS,
    sandbox="subprocess",
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=generation_constraints,
    adapter_manifest=ADAPTERS,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=str(GRAPHS_DIR),
    embeddings_dir=str(EMBEDDINGS_DIR),
    wheels_dir=WHEELS_COMPETITION,
    task_ids=list(selected),
    concurrency=CONCURRENCY,
    display_mode=DISPLAY,
    verbose=False,
)
accepted = accepted_fields(EvalConfig)
ignored = sorted(set(eval_kwargs) - accepted)
eval_kwargs = {key: value for key, value in eval_kwargs.items() if key in accepted}
if ignored:
    print("ignored by this EvalConfig:", ignored)

started = time.time()
evaluation = run_sync(Evaluator(EvalConfig(**eval_kwargs)).run())
elapsed = time.time() - started
print(
    f"evaluation complete: {evaluation.resolved}/{evaluation.total} resolved "
    f"({evaluation.resolution_rate:.2%}) in {elapsed:.0f}s"
)

## 8. Archive and report

In [ ]:
# =============================================================================
# 8. Archive + report — the same runs/<UTC>/ handoff the host trial produces.
# =============================================================================
if not RESULTS_DIR.is_dir():
    raise SystemExit("no results directory; run the evaluation cell first")

RUNS_ROOT = WORK / "artifacts" / "runs"
ARCHIVE_ENV = {
    **os.environ,
    "PYTHONPATH": str(REPO_DIR / "src"),
    "ROBOTINA_HERMES_MODEL": MODEL_ID,
    "ROBOTINA_HERMES_MODEL_BASE_URL": BASE_URL,
}
ARCHIVE_ARGV = [
    sys.executable,
    "-m",
    "kaggle_gemma_agent.harness_runs",
    "archive",
    RESULTS_DIR,
    "--runs-root",
    RUNS_ROOT,
    "--tasks",
    TASKS_FILE,
    "--backend",
    MODEL_ID,
    "--backend-name",
    BACKEND_NAME,
    "--model",
    MODEL_ID,
    "--sandbox-deps-mode",
    "faithful",
    "--no-journal",
]
if SUBSET:
    ARCHIVE_ARGV += ["--subset", SUBSET]

archived = sh(ARCHIVE_ARGV, cwd=REPO_DIR, env=ARCHIVE_ENV)
if archived.returncode != 0:
    raise SystemExit("archive failed; the raw results/ directory is still complete")

RUN_DIR = max(RUNS_ROOT.iterdir(), key=lambda path: path.stat().st_mtime)
print()
print((RUN_DIR / "STATUS").read_text(encoding="utf-8"))

records = [
    json.loads(line)
    for line in (RESULTS_DIR / "task_results.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"{'instance_id':<20}{'resolved':<10}{'tools':<8}error")
for record in records:
    print(
        f"{record.get('instance_id', '?'):<20}"
        f"{record.get('resolved')!s:<10}"
        f"{record.get('tool_calls')!s:<8}"
        f"{record.get('error') or ''}"
    )

## 9. Package the artifacts

In [ ]:
# =============================================================================
# 9. Package — one zip with results/, runs/ and the run metadata.
# =============================================================================
BUNDLE = Path("/kaggle/working") / f"{RUN_LABEL}.zip"
META = {
    "run_label": RUN_LABEL,
    "repo": REPO_URL,
    "repo_ref": REPO_REF,
    "repo_commit": COMMIT,
    "model_backend": MODEL_BACKEND,
    "model_id": MODEL_ID,
    "backend_name": BACKEND_NAME,
    "base_url": BASE_URL,
    "data_root": str(DATA),
    "task_ids": selected,
    "budgets": {
        "max_tool_calls": max_tool_calls,
        "max_time_minutes": max_time_minutes,
        "max_turns": max_turns,
        "command_timeout_seconds": timeout_seconds,
    },
    "resolved": evaluation.resolved,
    "total": evaluation.total,
    "archived_run": str(RUN_DIR),
}
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for root in (RESULTS_DIR, RUNS_ROOT):
        for path in root.rglob("*"):
            if path.is_file():
                bundle.write(path, path.relative_to(WORK))
    bundle.writestr("run_meta.json", json.dumps(META, indent=2))

print(f"{BUNDLE} ({BUNDLE.stat().st_size / 2**20:.1f} MiB)")
print("Download it from the notebook output panel.")

## 10. Stop the model server

In [ ]:
# =============================================================================
# 10. Stop the model server — always run this when you are done with the session.
# =============================================================================
if MODEL_SERVER is not None:
    MODEL_SERVER.stop()
    MODEL_SERVER = None
    print("vLLM stopped")
if HEADER_PROXY is not None:
    HEADER_PROXY.stop()
    HEADER_PROXY = None
    print("header proxy stopped")

## What to bring back

- `<RUN_LABEL>.zip` from the notebook output panel. It holds `results/<RUN_LABEL>/`
  (summary, task results, patches, test outputs, traces, logs), `artifacts/runs/<UTC>/`
  (`report.json`, `STATUS`, per-task artifacts) and `run_meta.json`.
- Copy the archive into the repository's `runs/` workflow on the host and analyse it
  with `docs/run-reports.md`; do not paste logs into chat.

## Reading the result

- `resolved` is the SWE-bench verdict for one task; the run rate is resolved / tasks.
- `STATUS` is the six-line handoff: backend, tasks/resolved/rate, budget use, failure
  kinds, delivery scope and artifact paths.
- A `test_failure` means the patch was applied and pytest failed; an `error` means the
  harness could not finish the task (snapshot, sandbox, model or budget). Read the
  per-task `error` before attributing anything to the harness.

## Known limits of this notebook

- The `subprocess` sandbox is not the 4 GiB / 2 vCPU Docker container; it isolates
  files and the process group, not memory or CPU.
- JUnit XML stays inside the sandbox and is lost on teardown, exactly as in a host
  trial, so the verdict comes from `task_results.jsonl`.
- The notebook image's Python and wheels may differ from the scored image; a wheel
  that is missing here is not necessarily missing at scoring time.
- `verification_timeout_seconds` and `scheduler_reserve_full_isl` exist in the
  official notebook's harness version but not in the pinned 0.2.11 wheelhouse; the
  notebook passes them only when the installed dataclass accepts them.
- The first remote run is evidence, not an estimate: a two-task run on `dev-v1` says
  nothing about the hidden test set.
